In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random
import os
import warnings

warnings.filterwarnings('ignore')

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("sklearn:", __import__('sklearn').__version__)
print("matplotlib:", __import__('matplotlib').__version__)
print("All imports successful.")

pandas: 3.0.2
numpy: 2.4.4
sklearn: 1.8.0
matplotlib: 3.10.9
All imports successful.


In [2]:
random.seed(42)
np.random.seed(42)
fake = Faker('en_IN')

CONFIG = {
    "total_users": 500,
    "total_transactions": 100000,
    "fraud_injection_rate": 0.025,
    "output_path": "../data/synthetic/transactions_raw.csv",
    "processed_path": "../data/processed/features.csv",
    "date_range_days": 90,
    "start_date": datetime(2024, 1, 1),
}

CITIES = [
    {"name": "Mumbai", "lat": 19.0760, "lon": 72.8777},
    {"name": "Delhi", "lat": 28.6139, "lon": 77.2090},
    {"name": "Bangalore", "lat": 12.9716, "lon": 77.5946},
    {"name": "Chennai", "lat": 13.0827, "lon": 80.2707},
    {"name": "Kolkata", "lat": 22.5726, "lon": 88.3639},
    {"name": "Hyderabad", "lat": 17.3850, "lon": 78.4867},
    {"name": "Pune", "lat": 18.5204, "lon": 73.8567},
    {"name": "Ahmedabad", "lat": 23.0225, "lon": 72.5714},
]

MERCHANT_CATEGORIES = [
    "grocery", "fuel", "utilities", "restaurant",
    "clothing", "electronics", "medical", "travel",
    "crypto", "international", "jewellery"
]

print("Configuration loaded.")
print("Total users:", CONFIG["total_users"])
print("Total transactions:", CONFIG["total_transactions"])
print("Fraud rate:", CONFIG["fraud_injection_rate"])

Configuration loaded.
Total users: 500
Total transactions: 100000
Fraud rate: 0.025


In [3]:
def generate_user_profiles(num_users):
    users = []
    for i in range(num_users):
        home_city = random.choice(CITIES)
        persona = random.choices(
            ["normal", "compromised", "fraudster"],
            weights=[0.92, 0.05, 0.03],
            k=1
        )[0]

        user = {
            "user_id": f"USR{str(i+1).zfill(5)}",
            "name": fake.name(),
            "email": fake.email(),
            "phone": fake.phone_number(),
            "home_city": home_city["name"],
            "home_lat": home_city["lat"],
            "home_lon": home_city["lon"],
            "persona": persona,
            "avg_transaction_amount": round(random.uniform(1000, 20000), 2),
            "typical_hour_start": random.randint(8, 11),
            "typical_hour_end": random.randint(18, 22),
            "primary_device": fake.uuid4(),
            "secondary_device": fake.uuid4() if random.random() > 0.5 else None,
            "preferred_categories": random.sample(MERCHANT_CATEGORIES, k=random.randint(2, 5)),
            "account_age_days": random.randint(180, 1800),
        }
        users.append(user)

    return pd.DataFrame(users)

user_profiles = generate_user_profiles(CONFIG["total_users"])
print("User profiles generated:", len(user_profiles))
print("Persona distribution:")
print(user_profiles["persona"].value_counts().to_string())

User profiles generated: 500
Persona distribution:
persona
normal         465
compromised     24
fraudster       11


In [4]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def generate_transaction(user, txn_id, timestamp, last_txn=None):
    city = random.choice(CITIES)
    amount = round(abs(np.random.normal(user["avg_transaction_amount"], user["avg_transaction_amount"] * 0.4)), 2)
    amount = max(50, amount)
    device = user["primary_device"]
    category = random.choice(user["preferred_categories"])
    is_fraud = 0

    if user["persona"] == "normal":
        if not (user["typical_hour_start"] <= timestamp.hour <= user["typical_hour_end"]):
            if random.random() > 0.05:
                hour = random.randint(user["typical_hour_start"], user["typical_hour_end"])
                timestamp = timestamp.replace(hour=hour)
        city_data = next(c for c in CITIES if c["name"] == user["home_city"])
        city = city_data

    elif user["persona"] == "compromised":
        if random.random() < 0.15:
            city = random.choice([c for c in CITIES if c["name"] != user["home_city"]])
            amount = round(user["avg_transaction_amount"] * random.uniform(4, 10), 2)
            device = fake.uuid4()
            timestamp = timestamp.replace(hour=random.randint(1, 4))
            category = random.choice(["crypto", "international", "jewellery"])
            is_fraud = 1

    elif user["persona"] == "fraudster":
        if random.random() < 0.40:
            amount = round(user["avg_transaction_amount"] * random.uniform(1.5, 3.0), 2)
            city = random.choice(CITIES)
            device = random.choice([user["primary_device"], fake.uuid4()])
            category = random.choice(["crypto", "international", "electronics", "jewellery"])
            is_fraud = 1

    km_from_last = 0
    minutes_from_last = 0
    impossible_speed = 0

    if last_txn is not None:
        km_from_last = round(haversine_distance(
            last_txn["lat"], last_txn["lon"], city["lat"], city["lon"]
        ), 2)
        minutes_from_last = max(1, round((timestamp - last_txn["timestamp"]).total_seconds() / 60, 2))
        speed_kmh = (km_from_last / minutes_from_last) * 60
        impossible_speed = 1 if speed_kmh > 900 else 0

    return {
        "txn_id": txn_id,
        "user_id": user["user_id"],
        "timestamp": timestamp,
        "amount": amount,
        "city": city["name"],
        "lat": city["lat"],
        "lon": city["lon"],
        "device_id": device,
        "merchant_category": category,
        "km_from_last_txn": km_from_last,
        "minutes_from_last_txn": minutes_from_last,
        "impossible_speed": impossible_speed,
        "is_fraud": is_fraud,
    }

print("Transaction generator defined.")

Transaction generator defined.


In [5]:
transactions = []
txn_id_counter = 1

user_profiles_list = user_profiles.to_dict("records")

for user in user_profiles_list:
    num_txns = random.randint(150, 250)
    last_txn = None

    current_date = CONFIG["start_date"] + timedelta(days=random.randint(0, 10))

    for _ in range(num_txns):
        current_date = current_date + timedelta(
            hours=random.randint(1, 24),
            minutes=random.randint(0, 59)
        )

        if current_date > CONFIG["start_date"] + timedelta(days=CONFIG["date_range_days"]):
            break

        txn = generate_transaction(
            user=user,
            txn_id=f"TXN{str(txn_id_counter).zfill(8)}",
            timestamp=current_date,
            last_txn=last_txn
        )

        transactions.append(txn)
        last_txn = txn
        last_txn["timestamp"] = current_date
        txn_id_counter += 1

df_raw = pd.DataFrame(transactions)
df_raw = df_raw.sort_values("timestamp").reset_index(drop=True)

print("Total transactions generated:", len(df_raw))
print("Fraud transactions:", df_raw["is_fraud"].sum())
print("Fraud percentage:", round(df_raw["is_fraud"].mean() * 100, 2), "%")
print("Date range:", df_raw["timestamp"].min(), "to", df_raw["timestamp"].max())

Total transactions generated: 78299
Fraud transactions: 1324
Fraud percentage: 1.69 %
Date range: 2024-01-01 02:01:00 to 2024-03-30 23:59:00


In [7]:
base_dir = r"C:\Users\Abhishek Shandilya\Desktop\suraksha"

output_path = os.path.join(base_dir, "data", "synthetic", "transactions_raw.csv")
user_path = os.path.join(base_dir, "data", "synthetic", "user_profiles.csv")

df_raw.to_csv(output_path, index=False)
user_profiles.to_csv(user_path, index=False)

print("Raw transactions saved:", output_path)
print("User profiles saved:", user_path)
print("Transactions shape:", df_raw.shape)
print("User profiles shape:", user_profiles.shape)

Raw transactions saved: C:\Users\Abhishek Shandilya\Desktop\suraksha\data\synthetic\transactions_raw.csv
User profiles saved: C:\Users\Abhishek Shandilya\Desktop\suraksha\data\synthetic\user_profiles.csv
Transactions shape: (78299, 13)
User profiles shape: (500, 15)


In [8]:
def engineer_features(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)

    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["is_night"] = ((df["hour"] >= 23) | (df["hour"] <= 4)).astype(int)

    df["txn_count_1hr"] = 0
    df["txn_count_24hr"] = 0
    df["amount_sum_1hr"] = 0.0
    df["amount_sum_24hr"] = 0.0
    df["amount_vs_user_avg"] = 0.0
    df["is_new_device"] = 0
    df["device_count_7d"] = 0
    df["is_new_merchant_category"] = 0

    user_groups = df.groupby("user_id")

    processed_chunks = []

    for user_id, group in user_groups:
        group = group.copy().reset_index(drop=True)
        user_avg = group["amount"].mean()
        seen_devices = set()
        seen_categories = set()

        for i, row in group.iterrows():
            current_time = row["timestamp"]

            past_1hr = group[
                (group["timestamp"] < current_time) &
                (group["timestamp"] >= current_time - timedelta(hours=1))
            ]
            past_24hr = group[
                (group["timestamp"] < current_time) &
                (group["timestamp"] >= current_time - timedelta(hours=24))
            ]
            past_7d = group[
                (group["timestamp"] < current_time) &
                (group["timestamp"] >= current_time - timedelta(days=7))
            ]

            group.at[i, "txn_count_1hr"] = len(past_1hr)
            group.at[i, "txn_count_24hr"] = len(past_24hr)
            group.at[i, "amount_sum_1hr"] = past_1hr["amount"].sum()
            group.at[i, "amount_sum_24hr"] = past_24hr["amount"].sum()
            group.at[i, "amount_vs_user_avg"] = round(row["amount"] / user_avg, 4) if user_avg > 0 else 1.0
            group.at[i, "is_new_device"] = 0 if row["device_id"] in seen_devices else 1
            group.at[i, "device_count_7d"] = past_7d["device_id"].nunique()
            group.at[i, "is_new_merchant_category"] = 0 if row["merchant_category"] in seen_categories else 1

            seen_devices.add(row["device_id"])
            seen_categories.add(row["merchant_category"])

        processed_chunks.append(group)

    df_featured = pd.concat(processed_chunks).reset_index(drop=True)
    return df_featured

print("Starting feature engineering. This will take 3-5 minutes.")
df_featured = engineer_features(df_raw)
print("Feature engineering complete.")
print("Shape:", df_featured.shape)
print("Columns:", list(df_featured.columns))

Starting feature engineering. This will take 3-5 minutes.
Feature engineering complete.
Shape: (78299, 25)
Columns: ['txn_id', 'user_id', 'timestamp', 'amount', 'city', 'lat', 'lon', 'device_id', 'merchant_category', 'km_from_last_txn', 'minutes_from_last_txn', 'impossible_speed', 'is_fraud', 'hour', 'day_of_week', 'is_weekend', 'is_night', 'txn_count_1hr', 'txn_count_24hr', 'amount_sum_1hr', 'amount_sum_24hr', 'amount_vs_user_avg', 'is_new_device', 'device_count_7d', 'is_new_merchant_category']


In [9]:
print("Null values per column:")
print(df_featured.isnull().sum().to_string())
print()
print("Fraud vs Non-Fraud feature comparison:")
comparison = df_featured.groupby("is_fraud")[
    ["amount_vs_user_avg", "km_from_last_txn", "is_new_device", "txn_count_1hr", "is_night"]
].mean().round(4)
print(comparison.to_string())
print()
print("Impossible speed events:", df_featured["impossible_speed"].sum())
print("Night transactions:", df_featured["is_night"].sum())
print("New device transactions:", df_featured["is_new_device"].sum())

Null values per column:
txn_id                      0
user_id                     0
timestamp                   0
amount                      0
city                        0
lat                         0
lon                         0
device_id                   0
merchant_category           0
km_from_last_txn            0
minutes_from_last_txn       0
impossible_speed            0
is_fraud                    0
hour                        0
day_of_week                 0
is_weekend                  0
is_night                    0
txn_count_1hr               0
txn_count_24hr              0
amount_sum_1hr              0
amount_sum_24hr             0
amount_vs_user_avg          0
is_new_device               0
device_count_7d             0
is_new_merchant_category    0

Fraud vs Non-Fraud feature comparison:
          amount_vs_user_avg  km_from_last_txn  is_new_device  txn_count_1hr  is_night
is_fraud                                                                              
0           

In [10]:
processed_path = os.path.join(base_dir, "data", "processed", "features.csv")

df_featured.to_csv(processed_path, index=False)

print("Processed data saved:", processed_path)
print("Shape:", df_featured.shape)
print("Fraud transactions:", df_featured["is_fraud"].sum())
print("Non-fraud transactions:", (df_featured["is_fraud"] == 0).sum())

Processed data saved: C:\Users\Abhishek Shandilya\Desktop\suraksha\data\processed\features.csv
Shape: (78299, 25)
Fraud transactions: 1324
Non-fraud transactions: 76975


In [3]:
base_dir = r"C:\Users\Abhishek Shandilya\Desktop\suraksha"

df_featured = pd.read_csv(os.path.join(base_dir, "data", "processed", "features.csv"))
df_featured["timestamp"] = pd.to_datetime(df_featured["timestamp"])

user_profiles = pd.read_csv(os.path.join(base_dir, "data", "synthetic", "user_profiles.csv"))

print("Processed data loaded:", df_featured.shape)
print("User profiles loaded:", user_profiles.shape)
print("Fraud transactions:", df_featured["is_fraud"].sum())
print("Date range:", df_featured["timestamp"].min(), "to", df_featured["timestamp"].max())

Processed data loaded: (78299, 25)
User profiles loaded: (500, 15)
Fraud transactions: 1324
Date range: 2024-01-01 02:01:00 to 2024-03-30 23:59:00
